In [2]:
import sparql_virtualizer
import rdflib

g = rdflib.Graph()


query = """
PREFIX ogc: <http://www.ogc.org/>
PREFIX ine: <https://lod.ine.es/voc/cubes/vocabulary#>
PREFIX sdmx-measure: <http://purl.org/linked-data/sdmx/2009/measure#>
PREFIX sdmx-dimension: <http://purl.org/linked-data/sdmx/2009/dimension#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX geof: <http://www.opengis.net/def/function/geosparql/>
PREFIX ex: <http://example.org/function/>
PREFIX qb: <http://purl.org/linked-data/cube#>

SELECT DISTINCT ?s ?gs ?y ?gy WHERE {
    ?s a ogc:administrativeunit ;
       ogc:nameunit "Santiago de Compostela" ;
       geo:hasGeometry ?gs .
    ?y a ogc:watercourselinksequence ;
        geo:hasGeometry ?gy .
    FILTER (geof:sfWithin(?gy, ?gs))
}
"""

In [6]:
import folium
import ast
from IPython.display import display

# 1. Configuración base
# Usamos un mapa base oscuro para que los bordes de colores resalten más
m = folium.Map(location=[40.41, -3.70], zoom_start=6, tiles='CartoDB positron')
puntos_para_encuadre = []

# Paleta de colores más brillantes para que destaquen sobre fondo oscuro y con bordes
colores_paleta = ['#3498db', '#e74c3c', '#2ecc71', '#f1c40f', '#9b59b6', '#e67e22']

qres = g.query(query)

for row in qres:
    # Iteramos sobre la fila de 2 en 2
    for i in range(0, len(row), 2):
        try:
            uri = str(row[i])
            raw_geom = str(row[i+1])
            
            if not raw_geom.startswith("{"):
                continue

            geom_dict = ast.literal_eval(raw_geom)
            
            # Selección de color
            color_index = (i // 2) % len(colores_paleta)
            color_actual = colores_paleta[color_index]
            
            # CONFIGURACIÓN DE ESTILO PARA EVITAR OPACIDAD
            # Usamos una opacidad de relleno MUY baja (0.1 o 0.2)
            # Pero una opacidad de borde ALTA (1.0) y un peso de línea mayor
            def style_function(feature, color=color_actual):
                return {
                    'fillColor': color,   # Color de relleno
                    'fillOpacity': 0.01,   # MUY TRANSPARENTE por dentro (15%)
                    'color': color,       # Color del borde
                    'opacity': 1.0,        # BORDE TOTALMENTE OPACO
                    'weight': 3,          # BORDE GRUESO para que se vea bien el contorno
                }
            
            # Añadir al mapa con el estilo modificado
            folium.GeoJson(
                geom_dict,
                tooltip=f"<b>Variable {i//2 + 1}:</b> {uri}",
                style_function=style_function
            ).add_to(m)
            
            # Recolectar coordenadas para auto-zoom
            if 'coordinates' in geom_dict:
                coords = geom_dict['coordinates']
                if geom_dict['type'] == 'Polygon':
                    # Manejo de Polígonos y MultiPolígonos si fuera necesario
                    p = coords[0][0] # Primer punto del primer anillo
                    puntos_para_encuadre.append([p[1], p[0]])
                elif geom_dict['type'] == 'Point':
                    puntos_para_encuadre.append([coords[1], coords[0]])
                    
        except Exception:
            # Capturar errores de parsing de geometría o valores nulos
            continue

# 2. Ajuste de vista
if puntos_para_encuadre:
    m.fit_bounds(puntos_para_encuadre)

display(m)

https://api-features.ign.es//collections/administrativeunit/items?f=json&limit=10000&nameunit=Santiago+de+Compostela
https://api-features.idee.es/collections/watercourselinksequence/items?f=json&limit=10000&bbox=-8.632011636%2C42.824144224%2C-8.389642519%2C42.989630989000005
